# 🔄 Enterprise VPS Security AI — Safe Retraining Notebook

This notebook retrains the model with **administrator-approved** unknown attacks.

### Workflow:
1. Load `enterprise_security_dataset.csv` (main dataset)
2. Load `reviewed_unknown_attacks.csv` (admin-approved new samples)
3. Merge, deduplicate, and retrain
4. Evaluate and export updated `attack_model.zip`

> **IMPORTANT**: Compatible with Kaggle, Google Colab, or Local environments.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets torch accelerate scikit-learn seaborn matplotlib pandas numpy

In [ ]:
import os, torch, json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# Step 2: Load datasets (Universal Kaggle, Colab, Local)
def find_file(filename):
    paths = [filename, f'./{filename}', f'/content/{filename}']
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                if f == filename or f.endswith(filename):
                    paths.insert(0, os.path.join(root, f))
    for p in paths:
        if os.path.exists(p):
            return p
    try:
        from google.colab import files
        print(f'Please upload {filename}:')
        uploaded = files.upload()
        return filename
    except (ImportError, Exception):
        return None

main_path = find_file('enterprise_security_dataset.csv')
new_path = find_file('reviewed_unknown_attacks.csv')

if not main_path or not os.path.exists(main_path):
    raise FileNotFoundError('enterprise_security_dataset.csv not found!')

df_main = pd.read_csv(main_path).dropna(subset=['text', 'label'])
print(f'Main dataset: {len(df_main)} samples, {df_main["label"].nunique()} classes')

if new_path and os.path.exists(new_path):
    df_new = pd.read_csv(new_path).dropna(subset=['text', 'label'])
    if 'status' in df_new.columns:
        df_new = df_new[df_new['status'] == 'approved']
    df_new = df_new[['text', 'label']]
    print(f'New approved samples: {len(df_new)}')
    print(f'New labels: {df_new["label"].unique().tolist()}')
    df_combined = pd.concat([df_main, df_new], ignore_index=True)
else:
    print('No new reviewed_unknown_attacks.csv found. Retraining on main dataset.')
    df_combined = df_main.copy()

df_combined = df_combined.drop_duplicates(subset=['text']).reset_index(drop=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nCombined dataset: {len(df_combined)} samples, {df_combined["label"].nunique()} classes')
print(df_combined['label'].value_counts())

In [ ]:
# Step 3: Label encoding
le = LabelEncoder()
df_combined['label_id'] = le.fit_transform(df_combined['label'])
all_labels = le.classes_.tolist()
num_labels = len(all_labels)
id2label = {i: l for i, l in enumerate(all_labels)}
label2id = {l: i for i, l in enumerate(all_labels)}
print(f'Total classes: {num_labels}')

In [ ]:
# Step 4: Tokenize and split
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, temp_df = train_test_split(df_combined, test_size=0.20, random_state=42, stratify=df_combined['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=256, padding=False)

def make_ds(dataframe):
    ds = Dataset.from_pandas(dataframe[['text', 'label_id']].rename(columns={'label_id': 'labels'}))
    return ds.map(tokenize_fn, batched=True, remove_columns=['text'])

train_dataset = make_ds(train_df)
val_dataset = make_ds(val_df)
test_dataset = make_ds(test_df)
print(f'Split: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}')

In [ ]:
# Step 5: Load model and retrain
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./retrain_results',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f}

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print('Starting retraining...')
trainer.train()
print('Retraining complete!')

In [ ]:
# Step 6: Evaluate
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

acc = accuracy_score(true_labels, preds)
p, r, f, _ = precision_recall_fscore_support(true_labels, preds, average='weighted', zero_division=0)

print(f'Accuracy  : {acc*100:.2f}%')
print(f'Precision : {p*100:.2f}%')
print(f'Recall    : {r*100:.2f}%')
print(f'F1-Score  : {f*100:.2f}%')

pred_names = [id2label[i] for i in preds]
true_names = [id2label[i] for i in true_labels]
print('\n' + classification_report(true_names, pred_names, zero_division=0))

In [ ]:
# Step 7: Save and export
import shutil

SAVE_DIR = './trained_model'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id, 'all_labels': all_labels}, f, indent=2)

# Update the main dataset
df_combined.to_csv('enterprise_security_dataset.csv', index=False)
print(f'Updated dataset saved: {len(df_combined)} samples')

# Export ZIP
zip_path = shutil.make_archive('attack_model', 'zip', '.', 'trained_model')
print(f'Model exported: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)')

try:
    from google.colab import files
    files.download('attack_model.zip')
    files.download('enterprise_security_dataset.csv')
except (ImportError, Exception):
    print('\n--> Running on Kaggle / Local environment.')
    print('    You can download "attack_model.zip" directly from the Output/Files section in the right sidebar!')